<a href="https://colab.research.google.com/github/Rijan-Ja-Bhattarai/AI-Girlfriend/blob/main/LLM1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x7a21b4169550>)

In [83]:
with open(file_path, "r", encoding="UTF-8") as f:
  text = f.read()
print(f"Total characters: {len(text):,}")
vocab = sorted(list(set(text)))
print(f"Vocabulary: {len(vocab)}")

Total characters: 20,479
Vocabulary: 62


In [84]:
from importlib_metadata import version
print(f"version: {version("tiktoken")}")

version: 0.14.0


In [85]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [86]:
input_text = "Adgsjkdhf sddfds#@$%^#&*"
encoded = tokenizer.encode(input_text)
print(f"Encoded: {encoded}")
decoded = tokenizer.decode(encoded)
print(f"Decoded: {decoded}")
print(f"Was it able to recontrust?: {input_text == decoded}")

Encoded: [2782, 14542, 73, 74, 34985, 69, 264, 1860, 69, 9310, 2, 31, 3, 4, 61, 2, 5, 9]
Decoded: Adgsjkdhf sddfds#@$%^#&*
Was it able to recontrust?: True


In [87]:
encoded_text = tokenizer.encode(text)
print(f"Token Size: {len(encoded_text):,}")

Token Size: 5,145


In [88]:
sample = encoded_text[50:]
context_size = 4 # How many tokens are included in the input for determining the targer
x = sample[:context_size]
y = sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [89]:
"""
For next word prediction the context should always be one word less than target
"""
for i in range(1, context_size+1):
  x = sample[:i]
  y = sample[i]
  print(f"When context is {x} target is {y}")

print("="*120)
for i in range(1, context_size+1):
  x = tokenizer.decode(sample[:i])
  y = tokenizer.decode([sample[i]])
  print(f"{x} -----> {y}")

When context is [290] target is 4920
When context is [290, 4920] target is 2241
When context is [290, 4920, 2241] target is 287
When context is [290, 4920, 2241, 287] target is 257
 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


In [90]:
import torch
from torch.utils.data import DataLoader, Dataset
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device being used: {device}")
print(f"Pytorch version: {torch.__version__}")
print(f"Cuda version: {torch.__version__[-3:-1:]}")
print("Import complete")

Device being used: cuda
Pytorch version: 2.11.0+cu130
Cuda version: 13
Import complete


In [91]:
from typing import Tuple

class GPTDataset(Dataset):
  def __init__(self, text: str, tokenizer: tiktoken.core.Encoding, max_length: int, stride: int) -> None:
    super().__init__()

    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(text) # Tokenizes the entire text

    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:max_length + i] # Sliding window
      target_chunk = token_ids[i+1:max_length + i + 1] # must always be one token ahead of the input chunk
      # Input ids and target ids are stored as tensors for effecient computing and storage
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, index) -> Tuple:
    return self.input_ids[index], self.target_ids[index]


# define the dataloader
def createDataloader(text: str,batch_size: int = 4, shuffle: bool = True, stride:int = 128,
                     drop_last: bool = True, max_length: int = 256, num_workers: int = 0):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDataset(text, tokenizer, max_length, stride)
  dataloader = DataLoader(dataset, batch_size, shuffle, num_workers=num_workers, drop_last=drop_last)
  return dataloader

In [92]:
dataloader = iter(createDataloader(text))
first_batch = next(dataloader)
second_batch = next(dataloader)
print(first_batch)
print(second_batch)

[tensor([[ 6164,    25,   366,  ...,    11,  4844,   286],
        [  520,  5493,  6776,  ...,   278,   465,  5101],
        [  198,  1544, 13818,  ..., 13476,   286,   616],
        [ 1310,  1165,   881,  ...,    13,   366,  2215]]), tensor([[   25,   366, 16773,  ...,  4844,   286,   262],
        [ 5493,  6776,   878,  ...,   465,  5101, 11061],
        [ 1544, 13818,  4622,  ...,   286,   616, 12036],
        [ 1165,   881, 40642,  ...,   366,  2215,   673]])]
[tensor([[ 4150,     8,  3688,  ..., 14093,   656,   465],
        [   11, 17728,   257,  ...,   422,   465,  2330],
        [  423,  4750,   326,  ...,   262,  8216,    13],
        [ 3347, 27846,   503,  ...,  2951,   625,     0]]), tensor([[    8,  3688,   284,  ...,   656,   465,  1021],
        [17728,   257,  8500,  ...,   465,  2330, 22645],
        [ 4750,   326,  9074,  ...,  8216,    13,   314],
        [27846,   503,  2048,  ...,   625,     0, 19672]])]


In [93]:
token_embedding_layer = torch.nn.Embedding(50257, 756)
print(token_embedding_layer)

Embedding(50257, 756)


In [97]:
dataloader = iter(createDataloader(text, batch_size=8, stride=4, max_length=4))
inputs, target = next(dataloader)
print(inputs)
print(f"Inputs shape: {inputs.shape}")
print(target)
print(f"Target Shape: {target.shape}")
print("="*60)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings)
print(f"Input token embeddings: {token_embeddings.shape}")

tensor([[   11,  1231, 35987,    11],
        [  319, 12036,    26,   475],
        [ 7521,    13,   843,   703],
        [40559, 11959,  1636,    11],
        [  703,  6655,   290, 10607],
        [   13,   402,   271, 10899],
        [  339,   373, 21840,    13],
        [ 7808,   607, 10927,  1108]])
Inputs shape: torch.Size([8, 4])
tensor([[ 1231, 35987,    11,   290],
        [12036,    26,   475,   340],
        [   13,   843,   703,   339],
        [11959,  1636,    11,   508],
        [ 6655,   290, 10607,   314],
        [  402,   271, 10899,   338],
        [  373, 21840,    13,   198],
        [  607, 10927,  1108,    26]])
Target Shape: torch.Size([8, 4])
tensor([[[-0.4876, -0.7464, -0.8458,  ..., -0.6085,  0.8230,  1.5874],
         [-0.2443, -0.4566,  1.1515,  ..., -0.6114,  0.7428, -1.4816],
         [-2.3728,  1.1848, -1.1852,  ..., -1.2221,  0.1729,  0.4961],
         [-0.4876, -0.7464, -0.8458,  ..., -0.6085,  0.8230,  1.5874]],

        [[ 2.0024,  1.0537,  2.4643,  

In [98]:
context_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, 756)
pos_embedding = pos_embedding_layer(torch.arange(context_length))
print(pos_embedding)
print(pos_embedding.shape)
input_embeddings = token_embeddings + pos_embedding
print(input_embeddings)

tensor([[-0.9245, -0.2324, -1.5285,  ...,  0.8348, -0.5180,  1.3656],
        [-0.4902,  0.9615, -1.3917,  ...,  1.1710,  0.0142, -0.6703],
        [ 0.4015, -0.6669, -0.2975,  ...,  0.4688,  0.3438,  0.0216],
        [-1.1537, -0.3670, -0.1644,  ...,  0.6021, -1.4309, -0.2476]],
       grad_fn=<EmbeddingBackward0>)
torch.Size([4, 756])
tensor([[[-1.4121, -0.9789, -2.3743,  ...,  0.2263,  0.3050,  2.9530],
         [-0.7345,  0.5049, -0.2402,  ...,  0.5596,  0.7570, -2.1519],
         [-1.9713,  0.5179, -1.4827,  ..., -0.7533,  0.5167,  0.5177],
         [-1.6413, -1.1135, -1.0102,  ..., -0.0064, -0.6079,  1.3398]],

        [[ 1.0779,  0.8213,  0.9357,  ...,  0.9351, -2.3651,  2.1504],
         [-0.0095, -0.4889, -0.4746,  ...,  1.8031, -2.3394,  2.8709],
         [ 0.2464, -1.0458, -1.1603,  ...,  1.3864,  0.3271, -0.8350],
         [-2.1113,  0.2172,  0.5440,  ...,  0.3727, -1.5342, -1.4926]],

        [[-2.0902, -0.1687, -1.7964,  ...,  1.3873, -1.2383,  1.0711],
         [-0.5906,

In [100]:
inputs = torch.tensor(
  [
    [0.43, 0.15, 0.89], # Your (x^1)
    [0.55, 0.87, 0.66], # journey (x^2)
    [0.57, 0.85, 0.64], # starts (x^3)
    [0.22, 0.58, 0.33], # with (x^4)
    [0.77, 0.25, 0.10], # one (x^5)
    [0.05, 0.80, 0.55]
  ] # step (x^6)
)

In [108]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print(f"Attention weights: {attn_weights_2_tmp}")
print(f"Sum: {attn_weights_2_tmp.sum()}")

attention_weights_2 = torch.softmax(attn_scores_2, dim=0)
print(f"Attention Weights (Softmax): {attention_weights_2}")
print(f"Sum: {attention_weights_2.sum()}")

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: 1.0000001192092896
Attention Weights (Softmax): tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: 1.0


In [113]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape[0])
for i, x_i in enumerate(inputs):
  context_vec_2 += attention_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [118]:
attn_scores = inputs @ inputs.T # Each token is mapped to every other token

# Compute attention weights
attn_weights = torch.softmax(attn_scores, dim=0)

# Compute context vector
context_vecs = attn_weights @ inputs
print(attn_scores)
print(attn_weights)
print(context_vecs)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.2098, 0.1385, 0.1390, 0.1435, 0.1526, 0.1385],
        [0.2006, 0.2379, 0.2369, 0.2074, 0.1958, 0.2184],
        [0.1981, 0.2333, 0.2326, 0.2046, 0.1975, 0.2128],
        [0.1242, 0.1240, 0.1242, 0.1462, 0.1367, 0.1420],
        [0.1220, 0.1082, 0.1108, 0.1263, 0.1879, 0.0988],
        [0.1452, 0.1581, 0.1565, 0.1720, 0.1295, 0.1896]])
tensor([[0.4017, 0.5023, 0.5059],
        [0.5595, 0.7824, 0.6953],
        [0.5538, 0.7686, 0.6834],
        [0.3369, 0.4647, 0.4119],
        [0.3525, 0.4059, 0.3657],
        [0.3856, 0.5761, 0.5077]])


In [146]:
torch.manual_seed(123)
x_2 = inputs[1]
d_in = inputs.shape[1] # Input embedding size = 3
d_out = 2 # Output embedding size = 2
"""
In GPT models output and input embedding sizes are same but to better follow computation
we are using 3 and 2 for inputs and output embedding sizes respectively
"""
W_query = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# Require grad set to false to reduce visual clutter during model training these would be
# turned to requires_grad = True

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)
print(key_2)
print(value_2)

tensor([-1.1729, -0.0048])
tensor([0.3951, 1.0037])
tensor([1.3730, 1.5279])


In [147]:
query = inputs @ W_query
key = inputs @ W_key
value = inputs @ W_value
print(query)

tensor([[-1.1686,  0.2019],
        [-1.1729, -0.0048],
        [-1.1438, -0.0018],
        [-0.6339, -0.0439],
        [-0.2979,  0.0535],
        [-0.9596, -0.0712]])


In [148]:
# Attention score w22
attn_scores_22 = query_2 @ key_2
print(attn_scores_22)

tensor(-0.4683)


In [149]:
attn_scores_2 = query_2 @ key.T
print(attn_scores_2)

tensor([-0.2218, -0.4683, -0.4598, -0.2833, -0.1766, -0.3816])


In [150]:
d_k = key.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim=0)
print(attn_weights_2)

tensor([0.1796, 0.1509, 0.1518, 0.1719, 0.1854, 0.1604])


In [151]:
context_vec_2 = attn_weights_2 @ value
print(context_vec_2)

tensor([0.9768, 1.1317])


In [154]:
# Implementing a self attention class
import torch.nn as nn
class SelfAttention(nn.Module):
  def __init__(self, d_in: int, d_out: int):
    super().__init__()
    self.W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
    self.W_key = nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
    self.W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)

  def forward(self, x):
    queries = x @ self.W_query
    keys = x @ self.W_key
    values = x @ self.W_value

    # Attention Scores
    attn_scores = queries @ keys.T

    # Attention Weights
    d = keys.shape[-1]
    attn_weights = torch.softmax(
        attn_scores / d**0.5, dim=-1
    )

    # Context vector
    context_vec = attn_weights @ values
    return context_vec

In [155]:
torch.manual_seed(123)
sa_v1= SelfAttention(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [156]:
import torch.nn as nn
class SelfAttention_v2(nn.Module):
  def __init__(self, d_in: int, d_out: int, qkv_bias: bool=False) -> None:
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    # attention scores
    attn_scores = queries @ keys.T

    # attention weights
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=0)

    # context vector
    context_vec = attn_weights @ values

    return context_vec

In [158]:
sa_v2 = SelfAttention_v2(3, 2, False)
print(sa_v2(inputs))

tensor([[0.5278, 0.2474],
        [0.5188, 0.2431],
        [0.5190, 0.2432],
        [0.5569, 0.2612],
        [0.5433, 0.2548],
        [0.5519, 0.2588]], grad_fn=<MmBackward0>)
